# Lab 1: Azure Prompt Flow — RAG Setup

**Duration:** 30 minutes  
**Environment:** Azure ML Studio Notebook  

---

## Learning Objectives

- Set up Azure AI Foundry project with Prompt Flow
- Deploy Azure OpenAI models (gpt-4o + text-embedding-3-small)
- Create Azure AI Search service
- Verify all connections for RAG pipeline

## Prerequisites

- Azure subscription with AI Foundry access
- Contributor role on the resource group

---

© 2026 Great Learning. All rights reserved.

> **💰 Cost-Optimized for Training**
> 
> This lab uses minimum-viable Azure resources:
> - **GPT-4o** with **5K TPM** (minimum capacity, sufficient for lab exercises)
> - **10K TPM** for embeddings (minimum for indexing)
> - **Free tier** Azure AI Search (sufficient for <3 indexes)
> - **Standard_LRS** storage (cheapest redundancy)
> - **max_tokens capped** at 50-150 across all labs
> 
> Estimated cost: **~$3-5/hour** while actively running labs.  
> Run the cleanup cell in Lab 9 when done to stop all charges.

## Step 1: Set Environment Variables

These variables will be used across all 4 labs. The random suffix ensures unique resource names.

In [ ]:
import os, random

RESOURCE_GROUP = "rg-promptflow-rag-lab"
LOCATION = "eastus"
AI_HUB_NAME = "ai-hub-banking-lab"
AI_PROJECT_NAME = "banking-rag-project"
OPENAI_NAME = f"aoai-banking-lab-{random.randint(1000, 9999)}"
SEARCH_NAME = f"search-banking-lab-{random.randint(1000, 9999)}"
STORAGE_NAME = f"stbankinglab{random.randint(10000, 99999)}"

# Store in environment for later cells
os.environ["RESOURCE_GROUP"] = RESOURCE_GROUP
os.environ["LOCATION"] = LOCATION
os.environ["OPENAI_NAME"] = OPENAI_NAME
os.environ["SEARCH_NAME"] = SEARCH_NAME
os.environ["STORAGE_NAME"] = STORAGE_NAME

print(f"Resource Group: {RESOURCE_GROUP}")
print(f"Location:       {LOCATION}")
print(f"OpenAI Name:    {OPENAI_NAME}")
print(f"Search Name:    {SEARCH_NAME}")
print(f"Storage Name:   {STORAGE_NAME}")

## Azure CLI Authentication

Azure ML compute instances have a **managed identity** — this logs in instantly with no browser needed.

> If managed identity fails (permissions not assigned), the cell falls back to device code login.

In [ ]:
import subprocess, json as _j

# Check if already logged in
r = subprocess.run(['az', 'account', 'show'], capture_output=True, text=True)
if r.returncode == 0:
    acct = _j.loads(r.stdout)
    print(f'✅ Already logged in: {acct.get("user", {}).get("name", "unknown")}')
    print(f'   Subscription: {acct.get("name", "unknown")}')
else:
    # Try managed identity first (instant on Azure ML compute)
    r2 = subprocess.run(['az', 'login', '--identity'], capture_output=True, text=True)
    if r2.returncode == 0:
        acct = _j.loads(r2.stdout)[0]
        print(f'✅ Logged in via managed identity')
        print(f'   Subscription: {acct.get("name", "unknown")}')
    else:
        print('Managed identity not available. Using device code login...')
        !az login --use-device-code

In [ ]:
# Verify and optionally set subscription
!az account show --output table

# Uncomment to set a specific subscription:
# !az account set --subscription "YOUR_SUBSCRIPTION_ID"

## Step 2: Create Resource Group

All lab resources will live in a single resource group for easy cleanup.

In [ ]:
!az group create --name {RESOURCE_GROUP} --location {LOCATION} --output table

**Expected output:** A table showing the resource group with `Succeeded` provisioning state.

## Step 3: Create Azure OpenAI Service

This creates the Azure OpenAI resource that will host our gpt-4o and embedding models.

In [ ]:
!az cognitiveservices account create \
    --name {OPENAI_NAME} \
    --resource-group {RESOURCE_GROUP} \
    --location {LOCATION} \
    --kind OpenAI \
    --sku S0 \
    --output table

**Expected output:** Table with the OpenAI resource showing `Succeeded` status. This may take 1-2 minutes.

## Step 4: Deploy gpt-4o Model (Cost-Optimized)

gpt-4o is our cost-optimized chat model (90% cheaper than gpt-4o) for generating RAG responses.

In [ ]:
# Deploy gpt-4o (5K TPM — sufficient for lab exercises)
# If you get InsufficientQuota, reduce --sku-capacity further
# or delete unused GPT-4o deployments in Azure Portal
!az cognitiveservices account deployment create \
    --name {OPENAI_NAME} \
    --resource-group {RESOURCE_GROUP} \
    --deployment-name "gpt-4o" \
    --model-name "gpt-4o" \
    --model-version "2024-08-06" \
    --model-format OpenAI \
    --sku-capacity 5 \
    --sku-name "Standard" \
    --output table

**Expected output:** Deployment details table showing `gpt-4o` model with Standard SKU.

## Step 5: Deploy Embedding Model

`text-embedding-3-small` generates 1536-dimensional vectors for our banking documents and queries.

In [ ]:
!az cognitiveservices account deployment create \
    --name {OPENAI_NAME} \
    --resource-group {RESOURCE_GROUP} \
    --deployment-name "text-embedding-3-small" \
    --model-name "text-embedding-3-small" \
    --model-version "1" \
    --model-format OpenAI \
    --sku-capacity 10 \
    --sku-name "Standard" \
    --output table

**Expected output:** Deployment details for `text-embedding-3-small` with 120 TPM capacity.

## Step 6: Create Azure AI Search Service

Azure AI Search will store our banking document index with vector and semantic search capabilities.

In [ ]:
!az search service create \
    --name {SEARCH_NAME} \
    --resource-group {RESOURCE_GROUP} \
    --location {LOCATION} \
    --sku free \
    --partition-count 1 \
    --replica-count 1 \
    --output table

**Expected output:** Search service details with Basic SKU. This may take 2-3 minutes to provision.

## Step 7: Create Storage Account

Storage will hold our banking documents before they are indexed.

In [ ]:
!az storage account create \
    --name {STORAGE_NAME} \
    --resource-group {RESOURCE_GROUP} \
    --location {LOCATION} \
    --sku Standard_LRS \
    --kind StorageV2 \
    --output table

**Expected output:** Storage account details with `Standard_LRS` redundancy.

## Step 8: Retrieve Keys and Endpoints

We need the endpoints and keys for all services. These will be saved for use in Labs 2-4.

In [ ]:
import subprocess, json

# Azure OpenAI endpoint
result = subprocess.run(
    ["az", "cognitiveservices", "account", "show",
     "--name", OPENAI_NAME, "--resource-group", RESOURCE_GROUP,
     "--query", "properties.endpoint", "-o", "tsv"],
    capture_output=True, text=True
)
OPENAI_ENDPOINT = result.stdout.strip()
# Ensure endpoint ends with /
if not OPENAI_ENDPOINT.endswith('/'):
    OPENAI_ENDPOINT += '/'

# Azure OpenAI key
result = subprocess.run(
    ["az", "cognitiveservices", "account", "keys", "list",
     "--name", OPENAI_NAME, "--resource-group", RESOURCE_GROUP,
     "--query", "key1", "-o", "tsv"],
    capture_output=True, text=True
)
OPENAI_KEY = result.stdout.strip()

# Azure AI Search
SEARCH_ENDPOINT = f"https://{SEARCH_NAME}.search.windows.net"

result = subprocess.run(
    ["az", "search", "admin-key", "show",
     "--service-name", SEARCH_NAME, "--resource-group", RESOURCE_GROUP,
     "--query", "primaryKey", "-o", "tsv"],
    capture_output=True, text=True
)
SEARCH_KEY = result.stdout.strip()

# Storage connection string
result = subprocess.run(
    ["az", "storage", "account", "show-connection-string",
     "--name", STORAGE_NAME, "--resource-group", RESOURCE_GROUP,
     "--query", "connectionString", "-o", "tsv"],
    capture_output=True, text=True
)
STORAGE_CONNECTION = result.stdout.strip()

# Store in environment for subsequent cells and labs
os.environ["OPENAI_ENDPOINT"] = OPENAI_ENDPOINT
os.environ["OPENAI_KEY"] = OPENAI_KEY
os.environ["SEARCH_ENDPOINT"] = SEARCH_ENDPOINT
os.environ["SEARCH_KEY"] = SEARCH_KEY
os.environ["STORAGE_CONNECTION"] = STORAGE_CONNECTION

print("Connection Details:")
print(f"  OpenAI Endpoint:  {OPENAI_ENDPOINT}")
print(f"  Search Endpoint:  {SEARCH_ENDPOINT}")
print(f"  Storage Account:  {STORAGE_NAME}")
print(f"  Keys retrieved:   OK")

## Step 9: Test Azure OpenAI Connection

Let's verify gpt-4o is responding correctly with a banking-related question.

In [ ]:
import urllib.request, json, ssl, subprocess

ctx = ssl.create_default_context()

# Verify deployment exists using az CLI (reliable regardless of endpoint format)
r = subprocess.run(['az', 'cognitiveservices', 'account', 'deployment', 'list',
    '--name', OPENAI_NAME, '--resource-group', RESOURCE_GROUP,
    '--query', "[?name=='gpt-4o'].name", '-o', 'tsv'],
    capture_output=True, text=True)
if 'gpt-4o' not in r.stdout:
    print('ERROR: gpt-4o deployment not found!')
    print('   Go back to Step 4 and re-run the deployment command.')
    print('   If quota error: reduce --sku-capacity to 1 or delete unused deployments.')
    raise SystemExit('Fix Step 4 first, then re-run this cell.')
print('gpt-4o deployment confirmed via CLI')

url = f"{OPENAI_ENDPOINT}openai/deployments/gpt-4o/chat/completions?api-version=2024-06-01"
data = json.dumps({
    "messages": [
        {"role": "system", "content": "You are a banking assistant. Reply briefly."},
        {"role": "user", "content": "What is RAG in the context of AI?"}
    ],
    "max_tokens": 50,
    "temperature": 0.3
}).encode()

req = urllib.request.Request(url, data=data, headers={
    "Content-Type": "application/json",
    "api-key": OPENAI_KEY
})

with urllib.request.urlopen(req, context=ctx) as resp:
    result = json.loads(resp.read())

print(f'GPT-4o Response:')
print(result['choices'][0]['message']['content'])
print(f'\nAzure OpenAI connection verified.')

**Expected output:** A brief explanation of RAG (Retrieval-Augmented Generation) from gpt-4o.

## Step 10: Test Embedding Model

Verify the embedding model returns 1536-dimensional vectors.

In [ ]:
url = f"{OPENAI_ENDPOINT}openai/deployments/text-embedding-3-small/embeddings?api-version=2024-06-01"
data = json.dumps({
    "input": "What is the wire transfer limit for business accounts?"
}).encode()

req = urllib.request.Request(url, data=data, headers={
    "Content-Type": "application/json",
    "api-key": OPENAI_KEY
})

with urllib.request.urlopen(req, context=ctx) as resp:
    result = json.loads(resp.read())

embedding = result["data"][0]["embedding"]
print(f"Embedding dimensions: {len(embedding)}")
print(f"First 5 values: {embedding[:5]}")
print("\n✅ Embedding model working correctly.")

**Expected output:** 1536 dimensions with floating-point values.

## Step 11: Verify Azure AI Search

Confirm the search service is active and ready to accept indexes.

In [ ]:
import subprocess

# Verify Search service exists and is running via CLI first
r = subprocess.run(['az', 'search', 'service', 'show',
    '--name', SEARCH_NAME, '--resource-group', RESOURCE_GROUP,
    '--query', 'status', '-o', 'tsv'],
    capture_output=True, text=True)

if r.returncode != 0 or 'running' not in r.stdout.lower():
    print(f'ERROR: Search service {SEARCH_NAME} not found or not running!')
    print(f'  Status: {r.stdout.strip() if r.stdout else "not found"}')
    print('  Go back to Step 6 and wait for provisioning to complete.')
    raise SystemExit('Fix Step 6 first, then re-run this cell.')

print(f'Search service {SEARCH_NAME} confirmed via CLI (status: running)')

# Now verify via HTTP API
url = f"{SEARCH_ENDPOINT}/indexes?api-version=2024-07-01"
req = urllib.request.Request(url, headers={
    "Content-Type": "application/json",
    "api-key": SEARCH_KEY
})

try:
    with urllib.request.urlopen(req, context=ctx, timeout=10) as resp:
        result = json.loads(resp.read())
    print(f"Search service is active. Current indexes: {len(result.get('value', []))}")
    print("\n✅ Azure AI Search connection verified.")
except Exception as e:
    print(f"HTTP verification failed: {e}")
    print("But CLI verification passed — Search service is running.")
    print("\n✅ Azure AI Search verified via CLI (HTTP may need a moment to be ready).")

**Expected output:** `Current indexes: 0` (no indexes created yet — that's Lab 2).

## Step 12: Save Configuration for Subsequent Labs

Save all connection details so Labs 2-4 can load them automatically.

In [ ]:
from datetime import datetime

config = f"""# Lab Configuration — Generated {datetime.now().strftime('%Y-%m-%d %H:%M')}
export RESOURCE_GROUP="{RESOURCE_GROUP}"
export LOCATION="{LOCATION}"
export OPENAI_NAME="{OPENAI_NAME}"
export OPENAI_ENDPOINT="{OPENAI_ENDPOINT}"
export OPENAI_KEY="{OPENAI_KEY}"
export SEARCH_NAME="{SEARCH_NAME}"
export SEARCH_ENDPOINT="{SEARCH_ENDPOINT}"
export SEARCH_KEY="{SEARCH_KEY}"
export STORAGE_NAME="{STORAGE_NAME}"
export STORAGE_CONNECTION="{STORAGE_CONNECTION}"
"""

config_path = os.path.expanduser("~/lab-config.env")
with open(config_path, "w") as f:
    f.write(config)

print(f"✅ Configuration saved to {config_path}")
print("   Labs 2-4 will load this automatically.")

## ✅ Lab 1 Complete

**Resources created:**
- Azure OpenAI Service with gpt-4o + text-embedding-3-small
- Azure AI Search Service (Basic tier)
- Azure Storage Account
- All connections verified

**Next:** Open `promptflow_lab2_data_indexing.ipynb`


> 🧹 **Cleanup:** Run the cleanup cell in **Lab 9** when all labs are complete to delete all resources.
---

© 2026 Great Learning. All rights reserved.